# 0️⃣3️⃣ - Managing your Labels
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%9B%A0%EF%B8%8F%20Platform%20%2F%20IT%20Administrator-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Intermediate-yellow) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to create, assign, update, and delete labels for RBAC scoping on devices and remote users.

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

This playbook builds on concepts from:
- [01 — Managing your Remote Users](./01-manage-users.ipynb)
- [02 — Managing your Devices](./02-manage-devices.ipynb)

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to retrieve all labels configured on a RADKit service |
| 2 | How to create a new label with a custom name and hex color |
| 3 | How to assign labels to remote users and devices for RBAC scoping |
| 4 | How to update an existing label's name and color |
| 5 | How to delete a label by its ID |

### 🏷️ 🏷️ 🏷️ RBAC in RADKit
Role-based Access Control (RBAC) is built-in inside your RADKit service. However, it is not enabled by default. To activate it, navigate on the left menu to `RBAC labels` and toogle the button `Enable RBAC` on.

![RBAC enabled](../images/04-rbac-enabled.png)

Now, you can create labels and assign them to your remote users and devices. If a device has the same label as a remote user, this user will be enabled to interact with that device. This allows you to restrict how users interact with specific groups of devices in your inventory.

![RBAC users](../images/05-rbac-user.png)

![RBAC devices](../images/06-rbac-device.png)

---

## 🔍 Method 1: Retrieve Labels

**Best for:** Auditing the current label roster, verifying RBAC assignments, or looking up label IDs before assigning them to users or devices.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `list_labels()` to get all configured labels.
3. Each result is a label object with `id`, `name`, and `color` attributes.

**What you need:**
- The RADKit server address and superadmin password

### 1.1 List all labels

In [ ]:
from getpass import getpass
from IPython.display import display, HTML
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    my_labels = service.list_labels()
    for label in my_labels.root.result:
        swatch = f'<span style="display:inline-block;width:16px;height:16px;background:{label.color};border:1px solid #ccc;vertical-align:middle;margin-right:6px;border-radius:3px;"></span>'
        display(HTML(f'📌 Label: <b>{label.name}</b> (ID: {label.id}) {swatch} {label.color}'))

---

## ➕ Method 2: Create a Label

**Best for:** Provisioning new RBAC scopes for a lab environment, a project, or a user group that requires isolated device access.

**How it works:**
1. Instantiate a `NewLabel` object with a `name` and a `color` (lowercase hex, e.g. `#2a57b1`).
2. Pass it to `create_labels()` as a list.
3. Confirm creation by listing all labels.

> **Note:** The `color` value must be lowercase hex. Uppercase characters will cause label creation to fail.

### 2.1 Create a label and confirm the result

In [ ]:
from getpass import getpass
from IPython.display import display, HTML
from radkit_service.control_api import ControlAPI, NewLabel

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Creation of a new label
    service.create_labels(labels=[NewLabel(name="Lab", color="#2a57b1")])

    # Retrieval of all the labels
    my_labels = service.list_labels()
    for label in my_labels.root.result:
        swatch = f'<span style="display:inline-block;width:16px;height:16px;background:{label.color};border:1px solid #ccc;vertical-align:middle;margin-right:6px;border-radius:3px;"></span>'
        display(HTML(f'📌 Label: <b>{label.name}</b> (ID: {label.id}) {swatch} {label.color}'))

---

## 🏷️ Method 3: Assign Labels to Users and Devices

**Best for:** Scoping a remote user's access to a specific set of devices. For example, restricting a user to Lab devices for read-only operations.

**How it works:**
1. Look up the label ID with `list_labels()`.
2. Call `update_remote_user()` with `add_labels=[label_id]` to assign the label to a user.
3. Locate the target device UUID with `list_devices()`.
4. Call `update_device()` with an instance of the `UpdateDevice` class. This one must have an instance of `UpdateLabelSet` where you can replace, add or remove labels using the ID

> You can also remove labels by passing their IDs to the `remove_labels` parameter, or fully replace the set using `replace_labels`.

### 3.1 Assign a label to a user and a device

In [23]:
from getpass import getpass
from radkit_common.utils.formatting import to_canonical_name
from radkit_service.webserver.models.labels import UpdateLabelSet
from radkit_service.control_api import ControlAPI, UpdateDevice

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Get the id of our new label
    new_label_id = [label.id for label in service.list_labels().root.result if label.name == "Lab"][0]
    print(f"🏷️ 'Lab' label ID is: {new_label_id}")

    service.update_remote_user(
        "radkit_chatops.gen@cisco.com",
        add_labels=[new_label_id]
    )

    # Now, let's fetch the user's labels
    print(f"🏷️ User's label IDs are: { [label for label in service.get_remote_user(username='radkit_chatops.gen@cisco.com').result[0].labels]}")

    # Finally, we apply this tag to a specific device
    # Fetch the UUID of a device by its name
    device_uuid = None
    for device in service.list_devices().result:
        if device.name == to_canonical_name("p0-2e"):
            device_uuid = device.uuid

    if device_uuid is not None:
        # Updating the device with the new label set
        result = service.update_device(UpdateDevice(uuid=device_uuid, labelUpdate=UpdateLabelSet(add=[new_label_id])))

        # Printing the device information after the update
        my_device = service.get_device(device_uuid).root.result[0]
        print(f"🏷️ Device {my_device.name} label IDs after update are: {my_device.labels}")

🏷️ 'Lab' label ID is: 7
🏷️ User's label IDs are: [1, 4, 7]
🏷️ Device p0-2e label IDs after update are: {4, 7}


---

## ✏️ Method 4: Update a Label

**Best for:** Renaming a label or changing its color after it has been created and assigned, without needing to recreate it.

**How it works:**
1. Instantiate an `UpdateLabel` object with the target label's `id`, and the new `name` and/or `color`.
2. Pass it to `update_labels()` as a list.
3. Confirm the update by listing all labels.

> **Note:** The `color` value must be lowercase hex. The label `id` is required even when changing only the `name`.

### 4.1 Update a label's name and color

In [ ]:
from getpass import getpass
from IPython.display import display, HTML
from radkit_service.control_api import ControlAPI, UpdateLabel

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Update an existing label's name and color
    result = service.update_labels(labels=[UpdateLabel(id='6', name="Demo", color="#ee12b7")])

    # Retrieval of all the labels
    my_labels = service.list_labels()
    for label in my_labels.root.result:
        swatch = f'<span style="display:inline-block;width:16px;height:16px;background:{label.color};border:1px solid #ccc;vertical-align:middle;margin-right:6px;border-radius:3px;"></span>'
        display(HTML(f'📌 Label: <b>{label.name}</b> (ID: {label.id}) {swatch} {label.color}'))

---

## 🗑️ Method 5: Delete a Label

**Best for:** Decommissioning a label that is no longer needed, cleaning up after a demo, or removing a stale RBAC scope.

**How it works:**
1. Look up the label ID with `list_labels()` if you don't have it already.
2. Call `delete_labels(label_ids=[...])` with a list of IDs to remove.
3. Confirm removal by listing all labels.

### 5.1 Delete a label and confirm the result

In [ ]:
from getpass import getpass
from IPython.display import display, HTML
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Delete a label by ID
    service.delete_labels(label_ids=['6'])

    # Retrieval of all the labels
    my_labels = service.list_labels()
    for label in my_labels.root.result:
        swatch = f'<span style="display:inline-block;width:16px;height:16px;background:{label.color};border:1px solid #ccc;vertical-align:middle;margin-right:6px;border-radius:3px;"></span>'
        display(HTML(f'📌 Label: <b>{label.name}</b> (ID: {label.id}) {swatch} {label.color}'))

---

## 🔀 Which Method Should I Use?

| | 🔍 Retrieve | ➕ Create | 🏷️ Assign | ✏️ Update | 🗑️ Delete |
|---|---|---|---|---|---|
| **Function** | `list_labels()` | `create_labels()` | `update_remote_user()` / `update_device()` | `update_labels()` | `delete_labels()` |
| **Key parameter** | — | `NewLabel(name, color)` | `add_labels` / `replace_labels` / `remove_labels` | `UpdateLabel(id, name, color)` | `label_ids=[...]` |
| **Return type** | `LabelList` | — | — | — | — |
| **Affects users** | — | — | ✅ Yes | — | — |
| **Affects devices** | — | — | ✅ Yes | — | — |
| **Best for** | Auditing, ID lookup | Provisioning new scopes | RBAC enforcement | Renaming, recoloring | Cleanup, decommission |